# A/B Test Case Study: E-Commerce Checkout Optimization

This notebook demonstrates a complete A/B experiment lifecycle using the **AB Test Toolkit**.

We'll walk through:
1. **Hypothesis & Power Analysis** — designing the experiment with 80/20 allocation
2. **Data Generation** — synthetic data with realistic anomalies
3. **Integrity Checks** — SRM detection
4. **Statistical Analysis** — frequentist + Bayesian side-by-side
5. **Variance Reduction** — CUPED with pre-experiment covariate
6. **Time-Series Analysis** — detecting novelty effects
7. **Segmentation** — Simpson's Paradox detection
8. **Automated Recommendation** — Ship / No-Ship / Inconclusive


> **⚠️ Tutorial caveat — multi-dataset construction.** This notebook is a *teaching* walkthrough, not a production experiment. To make every diagnostic fire in a single run, several sections regenerate data with different anomalies injected:
>
> - **Sections 3–7** run on the main checkout dataset (novelty effect injected).
> - **Section 8 (Simpson's Paradox)** uses an independently generated dataset because Simpson's Paradox requires a specific composition imbalance.
> - **Section 10 (Automated Recommendation)** uses the burn-in-excluded main dataset.
>
> A real experiment runs each diagnostic against the **same** dataset and combines the signals once. Treat this notebook as an illustration of *what each tool looks like when it fires*, not as the canonical end-to-end pipeline. The Streamlit app's **Analyze Results** page runs the full pipeline against a single uploaded dataset.

In [ ]:
import numpy as np
import pandas as pd
from ab_test_toolkit.power import required_sample_size, power_curve
from ab_test_toolkit.frequentist import z_test
from ab_test_toolkit.bayesian import beta_binomial
from ab_test_toolkit.srm import check_srm
from ab_test_toolkit.cuped import cuped_adjust
from ab_test_toolkit.segmentation import segment_analysis
from ab_test_toolkit.recommendation import generate_recommendation, check_novelty
from ab_test_toolkit.data_generator import generate_experiment_data
from ab_test_toolkit.visualization import (
    mde_vs_n_curve, power_loss_curve, ci_comparison_plot,
    posterior_plot, segment_comparison_chart,
    cumulative_lift_chart, daily_treatment_effect,
)
print("✅ All modules loaded successfully")

## 1. Hypothesis & Domain Context

**Business Context**: Our e-commerce checkout team believes a simplified payment form will increase checkout completion rates.

**Hypothesis**: Reducing the number of form fields from 8 to 5 will increase checkout conversion rate by at least 2 percentage points.

- **Baseline rate**: 10% checkout completion
- **Minimum Detectable Effect (MDE)**: 2 percentage points (absolute)
- **Allocation**: 80/20 split (to limit exposure risk)
- **Significance level**: α = 0.05
- **Power**: 80%


In [ ]:
# Power analysis with 80% control / 20% treatment allocation (limits blast radius)
result = required_sample_size(
    baseline_rate=0.10,
    mde=0.02,
    alpha=0.05,
    power=0.80,
    allocation_ratio=0.25,  # 80% control / 20% treatment (limit blast radius)
    daily_traffic=5000,
)
print(f"Control group:   {result.n_control:,} users")
print(f"Treatment group: {result.n_treatment:,} users")
print(f"Total required:  {result.n_total:,} users")
print(f"Sample-size inflation vs balanced: {result.sample_inflation_pct:.1f}%")
print(f"Estimated duration: {result.estimated_days} days")

In [ ]:
# MDE vs Sample Size curve
pc = power_curve(baseline_rate=0.10, mde_range=(0.005, 0.05), allocation_ratio=0.25)
fig = mde_vs_n_curve(pc)
fig.show()

In [ ]:
# Sample-size inflation from unequal allocation
fig = power_loss_curve(baseline_rate=0.10, mde=0.02)
fig.show()

## 2. Synthetic Data Generation

We generate realistic experiment data with known anomalies for educational purposes:
- **Novelty effect**: inflated treatment effect in the first 3 days
- **Covariate**: pre-experiment browsing score (correlated with checkout behavior)


In [ ]:
# Generate data with novelty effect and covariate
data = generate_experiment_data(
    baseline_rate=0.10,
    effect_size=0.02,
    n_control=5000,
    n_treatment=5000,
    inject_novelty=True,
    novelty_days=3,
    novelty_multiplier=3.0,
    random_seed=42,
)
print(f"Dataset shape: {data.shape}")
print(f"Columns: {list(data.columns)}")
print(f"\nGroup sizes:")
print(data["group"].value_counts())
print(f"\nOverall conversion rate: {data['value'].mean():.4f}")

## 3. Integrity Check: Sample Ratio Mismatch

Before any analysis, we verify the randomization was clean.


In [ ]:
ctrl = data[data["group"] == "control"]
treat = data[data["group"] == "treatment"]

srm_result = check_srm(
    observed=(len(ctrl), len(treat)),
    expected_ratio=(0.5, 0.5),
)
print(f"Expected ratio: {srm_result.expected_ratio}")
print(f"Observed ratio: ({srm_result.observed_ratio[0]:.3f}, {srm_result.observed_ratio[1]:.3f})")
print(f"Chi² statistic: {srm_result.chi2_statistic:.4f}")
print(f"P-value: {srm_result.p_value:.4f}")
print(f"Mismatch detected: {srm_result.has_mismatch}")

## 4. Frequentist Analysis

Two-proportion Z-test with unpooled standard error.


In [ ]:
freq_result = z_test(ctrl["value"].values, treat["value"].values, alpha=0.05)
print(f"Test: {freq_result.test_type}")
print(f"Z-statistic: {freq_result.statistic:.4f}")
print(f"P-value: {freq_result.p_value:.4f}")
print(f"Point estimate: {freq_result.point_estimate:.4f}")
print(f"95% CI: [{freq_result.ci_lower:.4f}, {freq_result.ci_upper:.4f}]")
print(f"Cohen's h: {freq_result.effect_size:.4f}")
print(f"Significant at α=0.05: {freq_result.is_significant}")

In [ ]:
fig = ci_comparison_plot(freq_result)
fig.show()

## 5. Bayesian Analysis

Beta-Binomial conjugate model with uniform prior Beta(1,1).


In [ ]:
bayes_result = beta_binomial(ctrl["value"].values, treat["value"].values)
print(f"P(Treatment > Control): {bayes_result.prob_b_greater_a:.4f}")
print(f"Expected Loss: {bayes_result.expected_loss:.6f}")
print(f"95% Credible Interval: [{bayes_result.credible_interval[0]:.4f}, {bayes_result.credible_interval[1]:.4f}]")

In [ ]:
fig = posterior_plot(bayes_result)
fig.show()

## 6. CUPED Variance Reduction

Using the pre-experiment browsing score as a covariate to reduce variance.


In [ ]:
cuped_result = cuped_adjust(
    ctrl["value"].values,
    treat["value"].values,
    ctrl["covariate"].values,
    treat["covariate"].values,
)
print(f"Covariate correlation (ρ): {cuped_result.correlation:.4f}")
print(f"Variance reduction: {cuped_result.variance_reduction_pct:.1f}%")
print(f"\nUnadjusted estimate: {cuped_result.unadjusted_estimate:.4f}")
print(f"Unadjusted CI: [{cuped_result.unadjusted_ci[0]:.4f}, {cuped_result.unadjusted_ci[1]:.4f}]")
print(f"\nCUPED-adjusted estimate: {cuped_result.adjusted_estimate:.4f}")
print(f"Adjusted CI: [{cuped_result.adjusted_ci[0]:.4f}, {cuped_result.adjusted_ci[1]:.4f}]")
print(f"\nCI width reduction: {(1 - (cuped_result.adjusted_ci[1]-cuped_result.adjusted_ci[0])/(cuped_result.unadjusted_ci[1]-cuped_result.unadjusted_ci[0]))*100:.1f}%")

## 7. Time-Series Analysis: Detecting Novelty Effect

The early days of the experiment show an inflated treatment effect — a classic novelty effect.
We'll visualize the daily treatment effect and identify the burn-in period.


In [ ]:
fig = daily_treatment_effect(data)
fig.show()

In [ ]:
# Cumulative lift chart
fig = cumulative_lift_chart(data)
fig.show()

### Burn-in Exclusion

After identifying the novelty effect in the first 3 days, we exclude that period and re-analyze.


In [ ]:
# Exclude first 3 days
clean_data = data[data["day"] > 3].copy()
ctrl_clean = clean_data[clean_data["group"] == "control"]
treat_clean = clean_data[clean_data["group"] == "treatment"]

freq_clean = z_test(ctrl_clean["value"].values, treat_clean["value"].values)
print("After excluding novelty period (days 1-3):")
print(f"P-value: {freq_clean.p_value:.4f}")
print(f"Point estimate: {freq_clean.point_estimate:.4f}")
print(f"Significant: {freq_clean.is_significant}")

## 8. Segmentation & Simpson's Paradox

Let's generate a separate dataset specifically engineered to show Simpson's Paradox.


In [ ]:
# Generate data with Simpson's Paradox
simpsons_data = generate_experiment_data(
    baseline_rate=0.10,
    effect_size=0.02,
    n_control=3000,
    n_treatment=3000,
    inject_simpsons=True,
    random_seed=42,
)

seg_result = segment_analysis(simpsons_data)
print(f"Aggregate estimate: {seg_result.aggregate_estimate:.4f}")
print(f"Simpson's Paradox detected: {seg_result.simpsons_paradox}")
if seg_result.simpsons_details:
    print(f"Details: {seg_result.simpsons_details}")
print(f"\nPer-segment results:")
for seg in seg_result.segment_results:
    print(f"  {seg['segment']}: estimate={seg['estimate']:.4f}, p={seg['p_value']:.4f}, n={seg['n']}")

In [ ]:
fig = segment_comparison_chart(seg_result)
fig.show()

## 9. Peeking Illustration

Running a Z-test on day 3 data only — demonstrating how early peeking can lead to false positives.


In [ ]:
early_data = data[data["day"] <= 3]
ctrl_early = early_data[early_data["group"] == "control"]["value"].values
treat_early = early_data[early_data["group"] == "treatment"]["value"].values

if len(ctrl_early) > 10 and len(treat_early) > 10:
    early_result = z_test(ctrl_early, treat_early)
    print(f"Day 3 peek — P-value: {early_result.p_value:.4f}")
    print(f"Point estimate: {early_result.point_estimate:.4f}")
    print(f"Significant: {early_result.is_significant}")
    print("\n⚠️ Early peeking inflates false positive rate!")
    print("The novelty effect creates a misleadingly large treatment effect.")
else:
    print("Not enough early data for meaningful analysis")

## 10. Automated Recommendation

The recommendation engine synthesizes all diagnostic signals into a Ship / No-Ship / Inconclusive decision.


In [ ]:
# Compute novelty signal on the time-series dataset and combine all diagnostic
# results in the final recommendation. (seg_result was computed above on the
# Simpson's-paradox demo dataset; in a real experiment all signals would come
# from the same dataset.)
novelty_result = check_novelty(data)
print(f"Novelty detected: {novelty_result.has_novelty} (early/late ratio={novelty_result.ratio:.2f})\n")

recommendation = generate_recommendation(
    frequentist=freq_result,
    bayesian=bayes_result,
    srm=srm_result,
    segmentation=seg_result,
    novelty=novelty_result,
    practical_significance_threshold=0.005,
)
print(f"Recommendation: {recommendation.recommendation}")
print(f"\nReason: {recommendation.reason}")
print(f"\nFlags: {recommendation.flags}")
print(f"\nSupporting metrics:")
for k, v in recommendation.supporting_metrics.items():
    print(f"  {k}: {v}")

### How to read this recommendation

The decision combines five inputs: frequentist test, Bayesian posterior, SRM check, segmentation, and novelty diagnostic. Even when the headline `recommendation` is *Inconclusive* or *Investigate*, the `flags` list explains *why* — for example, novelty inflation, Simpson's reversal, or insufficient signal.

**Suggested next steps when each flag fires:**

| Flag | Recommended action |
|------|-------------------|
| Novelty effect detected | Re-run analysis excluding the burn-in window (see Section 7) before deciding |
| Simpson's Paradox | Decide at the segment level; do not act on the aggregate. Check segment definitions for confounding |
| SRM detected | **Stop.** Investigate randomization, logging, and assignment plumbing before trusting any other result |
| Twyman's Law (effect too good) | Validate metric definition, check for instrumentation bugs, look for outliers |
| Underpowered (Inconclusive) | Extend the experiment to hit the pre-registered sample size, or re-design with a smaller MDE |

## Summary

This case study demonstrated a complete A/B experiment lifecycle:

| Step | Tool | Finding |
|------|------|---------|
| Power Analysis | `required_sample_size` | Designed experiment with 80/20 split |
| SRM Check | `check_srm` | Verified clean randomization |
| Frequentist | `z_test` | Statistical significance assessment |
| Bayesian | `beta_binomial` | P(B > A) and expected loss |
| CUPED | `cuped_adjust` | Variance reduction with covariate |
| Time-Series | `daily_treatment_effect` | Detected novelty effect |
| Segmentation | `segment_analysis` | Simpson's Paradox detection |
| Recommendation | `generate_recommendation` | Automated decision |

The AB Test Toolkit provides a modular, transparent, and statistically rigorous framework for experimentation.
